# Chest X-Ray Pneumonia Classification — ResNet18 + Grad-CAM
**Run on Kaggle with GPU T4 enabled for fast training (~10 mins)**

Dataset: [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)

## 1. Install extra dependency

In [ ]:
!pip install grad-cam scikit-learn -q

## 2. Imports & Config

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
)

# Kaggle dataset path
DATA_ROOT   = "/kaggle/input/chest-xray-pneumonia/chest_xray"
TRAIN_DIR   = os.path.join(DATA_ROOT, "train")
VAL_DIR     = os.path.join(DATA_ROOT, "val")
TEST_DIR    = os.path.join(DATA_ROOT, "test")
MODEL_PATH  = "/kaggle/working/pneumonia_model.pth"

IMAGE_SIZE  = 224
BATCH_SIZE  = 32
NUM_EPOCHS  = 15
LR          = 1e-4
NUM_CLASSES = 2
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 3. Data Loaders

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=val_transforms)
test_dataset  = datasets.ImageFolder(TEST_DIR,  transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Classes : {train_dataset.classes}")
print(f"Train   : {len(train_dataset)} images")
print(f"Val     : {len(val_dataset)} images")
print(f"Test    : {len(test_dataset)} images")

## 4. Build Model (ResNet18 Transfer Learning)

In [ ]:
model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)
print("Model ready.")

## 5. Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

best_f1 = 0.0
history = {"loss": [], "val_f1": []}

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # --- Validate ---
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            preds = model(images.to(device)).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    val_f1   = f1_score(all_labels, all_preds, average="weighted")
    avg_loss = running_loss / len(train_loader)
    history["loss"].append(avg_loss)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}", end="")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), MODEL_PATH)
        print("  ✓ saved", end="")
    print()
    scheduler.step()

print(f"\nTraining complete. Best Val F1: {best_f1:.4f}")
print(f"Model saved to {MODEL_PATH}")

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["loss"],   marker="o"); ax1.set_title("Training Loss");   ax1.set_xlabel("Epoch")
ax2.plot(history["val_f1"], marker="o"); ax2.set_title("Validation F1");   ax2.set_xlabel("Epoch")
plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=150)
plt.show()

## 7. Evaluate on Test Set

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        probs   = torch.softmax(outputs, dim=1)[:, 1]
        preds   = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

print("Test Metrics")
print("-" * 30)
print(f"  Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
print(f"  Recall    : {recall_score(all_labels, all_preds):.4f}")
print(f"  Precision : {precision_score(all_labels, all_preds):.4f}")
print(f"  F1-Score  : {f1_score(all_labels, all_preds):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(all_labels, all_probs):.4f}")

## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(colormap="Blues")
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150)
plt.show()
print("Saved to /kaggle/working/confusion_matrix.png")

## 9. Download the trained model
After training, go to **Output** tab on the right → download `pneumonia_model.pth`  
Then place it in the `models/` folder of your local project.

In [ ]:
print("Files saved to /kaggle/working/:")
for f in os.listdir("/kaggle/working"):
    size = os.path.getsize(f"/kaggle/working/{f}") / 1e6
    print(f"  {f}  ({size:.1f} MB)")